# MOSAIC benchmark summary
Runtime, CPU, memory and I/O measurements from existing benchmark TSV files.
See the notes in the generated HTML report for interpretation and missing-data limitations.

In [ ]:
from pathlib import Path
import base64
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
from snakemake.io import regex
from IPython.display import display

root = Path(snakemake.params.benchmark_dir)
registry = list(snakemake.params.registry)
outputs = dict(snakemake.output.items())
for path in outputs.values():
    Path(path).parent.mkdir(parents=True, exist_ok=True)
metrics = ["s", "max_rss", "max_vms", "max_uss", "max_pss", "io_in", "io_out", "mean_load", "cpu_time"]
columns = ["benchmark_file", "measurement", "rule", "status", "message", "sample", "assembler", "sampling",
           "wildcards", "threads_declared_current", "input_bytes_existing_current", "input_files_measured_current",
           "wall_hours", "cpu_hours", "mean_cpu_cores"] + metrics
rows = []
patterns = [(entry, re.compile(regex(entry["pattern"]))) for entry in registry]
for path in sorted(root.rglob("*.tsv")):
    if path.relative_to(root).parts[0] == "benchmark_summary":
        continue
    entry, wc = None, {}
    for candidate, pattern in patterns:
        match = pattern.fullmatch(str(path))
        if match:
            entry, wc = candidate, match.groupdict()
            break
    base = dict(benchmark_file=str(path.relative_to(root)), rule=entry["name"] if entry else "legacy_unattributed",
                sample=wc.get("sample", wc.get("sample_nanopore", wc.get("sample_pacbio", ""))),
                assembler=wc.get("assembler", ""), sampling=wc.get("sampling", ""),
                wildcards=json.dumps(wc, sort_keys=True), threads_declared_current=entry["threads_declared"] if entry else None)
    # These are current accessible input sizes, not historical input measurements.
    existing = set()
    if entry:
        for template in entry["inputs"]:
            try:
                input_path = Path(template.format(**wc))
                if input_path.is_file():
                    existing.add(input_path)
            except (KeyError, ValueError):
                pass
    base["input_files_measured_current"] = len(existing)
    base["input_bytes_existing_current"] = sum(p.stat().st_size for p in existing) if existing else None
    try:
        frame = pd.read_csv(path, sep="\t", na_values=["-", "NA", "nan"])
        if frame.empty or not {"s", "max_rss", "cpu_time"}.issubset(frame.columns):
            raise ValueError("Empty benchmark or missing s/max_rss/cpu_time columns")
        for index, record in frame.iterrows():
            row = dict(base, measurement=index + 1, status="ok" if entry else "legacy", message="")
            for metric in metrics:
                row[metric] = pd.to_numeric(record.get(metric), errors="coerce")
            if pd.isna(row["s"]) or row["s"] < 0:
                row.update(status="invalid", message="Invalid elapsed time")
            row["wall_hours"] = row["s"] / 3600
            row["cpu_hours"] = row["cpu_time"] / 3600
            row["mean_cpu_cores"] = row["cpu_time"] / row["s"] if row["s"] > 0 else None
            rows.append(row)
    except (ValueError, OSError, pd.errors.ParserError) as error:
        rows.append(dict(base, measurement=None, status="invalid", message=str(error)))
jobs = pd.DataFrame(rows, columns=columns)
jobs.to_csv(outputs["jobs"], index=False)
valid = jobs[jobs["status"].isin(["ok", "legacy"])].copy()
for metric in metrics + ["wall_hours", "cpu_hours"]:
    valid[metric] = pd.to_numeric(valid[metric], errors="coerce")
summary = valid.groupby("rule", dropna=False).agg(
    measurements=("benchmark_file", "size"), benchmark_files=("benchmark_file", "nunique"),
    summed_job_hours=("wall_hours", lambda values: values.sum(min_count=1)),
    median_job_seconds=("s", "median"), max_job_seconds=("s", "max"),
    peak_rss_mb=("max_rss", "max"),
    summed_cpu_hours=("cpu_hours", lambda values: values.sum(min_count=1)),
    summed_io_in=("io_in", lambda values: values.sum(min_count=1)),
    summed_io_out=("io_out", lambda values: values.sum(min_count=1))).reset_index()
summary = summary.sort_values("summed_job_hours", ascending=False)
summary.to_csv(outputs["rules"], index=False)
display(summary)
display(jobs[jobs["status"] != "ok"][["benchmark_file", "status", "message"]])

fig, axes = plt.subplots(1, 2, figsize=(15, 8), layout="constrained")
for ax, metric, label in zip(axes, ["summed_job_hours", "peak_rss_mb"], ["Sum of job runtimes (hours)", "Largest sampled peak RSS (MB)"]):
    top = summary.dropna(subset=[metric]).nlargest(15, metric).sort_values(metric)
    if top.empty:
        ax.text(0.5, 0.5, "No valid benchmark measurements", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
    else:
        ax.barh(top["rule"], top[metric], color="#34759a")
        ax.set_xlabel(label)
        ax.grid(axis="x", alpha=0.2)
fig.suptitle("MOSAIC benchmark summary")
fig.savefig(outputs["figure"], dpi=150)
plt.show()
plt.close(fig)
image = base64.b64encode(Path(outputs["figure"]).read_bytes()).decode()
notes = """This report summarizes existing benchmark files across the results directory, potentially from multiple executions.
Summed job runtimes are not total workflow elapsed time; concurrent jobs overlap.
Repeated measurements remain separate rows. Invalid files are listed in the jobs table.
Legacy paths are not assigned to rules because old paths could be shared.
Missing or failed jobs cannot be inferred from absent benchmark files.
Thread settings and accessible input sizes come from the current workflow configuration, not historical execution.
Dynamic or deleted inputs may be absent from input sizes. CPU and memory are sampled process measurements;
peak RSS is not the total peak memory of concurrent workflow jobs."""
from html import escape
html = "<!doctype html><html><head><meta charset='utf-8'><title>MOSAIC benchmarks</title></head><body>"
html += "<h1>MOSAIC benchmark summary</h1><p>" + escape(notes) + "</p>"
html += "<p><a href='" + escape(Path(outputs["jobs"]).name) + "'>Job measurements CSV</a> | "
html += "<a href='" + escape(Path(outputs["rules"]).name) + "'>Rule summary CSV</a></p>"
html += "<img style='max-width:100%' src='data:image/png;base64," + image + "'>"
html += summary.to_html(index=False, escape=True)
html += "<h2>Files requiring attention</h2>" + jobs[jobs["status"] != "ok"][["benchmark_file", "status", "message"]].to_html(index=False, escape=True)
html += "</body></html>"
Path(outputs["html"]).write_text(html)
